In [1]:
import requests, re, json

ITEM_ID = "http://www.loc.gov/resource/sn87076601/1921-07-27/ed-1/?sp=1"
HEADERS = {"User-Agent": "AcademicResearch/1.0 (gender-bias-panel; mailto:shikhadodecha14@gmail.com)"}

m  = re.search(r"/resource/([^/]+)/(\d{4}-\d{2}-\d{2})/ed-(\d+)/", ITEM_ID)
sp = re.search(r"[?&]sp=(\d+)", ITEM_ID)
lccn, date, ed = m.group(1), m.group(2), m.group(3)
seq = sp.group(1) if sp else "1"
print(f"Parsed: lccn={lccn} date={date} ed={ed} seq={seq}\n")

def show(label, url, is_json=False):
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        print(f"[{label}] {url}")
        print(f"   status={r.status_code}  bytes={len(r.text)}")
        if r.status_code == 200:
            if is_json:
                data = r.json()
                hits = []
                def walk(o):
                    if isinstance(o, dict):
                        for v in o.values(): walk(v)
                    elif isinstance(o, list):
                        for v in o: walk(v)
                    elif isinstance(o, str):
                        lo = o.lower()
                        if lo.startswith("http") and (lo.endswith(".txt") or "ocr" in lo or "fulltext" in lo):
                            hits.append(o)
                walk(data)
                print(f"   top-level keys: {list(data.keys())[:12]}")
                print(f"   candidate fulltext/ocr URLs: {hits[:5] if hits else 'NONE'}")
            else:
                print(f"   first 200 chars: {r.text.strip()[:200]!r}")
    except Exception as e:
        print(f"[{label}] {url}\n   ERROR: {e}")
    print()

show("A: loc.gov JSON", f"http://www.loc.gov/resource/{lccn}/{date}/ed-{ed}/?sp={seq}&fo=json", is_json=True)
show("B: loc.gov item JSON", f"http://www.loc.gov/item/{lccn}/{date}/ed-{ed}/?fo=json", is_json=True)
show("C: chroniclingamerica ocr.txt", f"https://chroniclingamerica.loc.gov/lccn/{lccn}/{date}/ed-{ed}/seq-{seq}/ocr.txt")
show("D: loc.gov fulltext param", f"http://www.loc.gov/resource/{lccn}/{date}/ed-{ed}/?sp={seq}&fo=json&at=resource.fulltext", is_json=True)

Parsed: lccn=sn87076601 date=1921-07-27 ed=1 seq=1

[A: loc.gov JSON] http://www.loc.gov/resource/sn87076601/1921-07-27/ed-1/?sp=1&fo=json
   status=200  bytes=50897
   top-level keys: ['articles_and_essays', 'calendar_url', 'cite_this', 'fulltext_service', 'item', 'locations', 'method', 'more_like_this', 'next_resource', 'notice', 'options', 'page']
   candidate fulltext/ocr URLs: NONE

[B: loc.gov item JSON] http://www.loc.gov/item/sn87076601/1921-07-27/ed-1/?fo=json
   status=200  bytes=65830
   top-level keys: ['articles_and_essays', 'calendar_url', 'cite_this', 'item', 'locations', 'method', 'more_like_this', 'next_resource', 'notice', 'options', 'previous_resource', 'related_items']
   candidate fulltext/ocr URLs: NONE

[C: chroniclingamerica ocr.txt] https://chroniclingamerica.loc.gov/lccn/sn87076601/1921-07-27/ed-1/seq-1/ocr.txt
   status=403  bytes=5814

[D: loc.gov fulltext param] http://www.loc.gov/resource/sn87076601/1921-07-27/ed-1/?sp=1&fo=json&at=resource.fulltext
   ERR

In [2]:
import requests, json
HEADERS = {"User-Agent": "AcademicResearch/1.0 (gender-bias-panel; mailto:shikhadodecha14@gmail.com)"}
url = "http://www.loc.gov/resource/sn87076601/1921-07-27/ed-1/?sp=1&fo=json"
data = requests.get(url, headers=HEADERS, timeout=30).json()

print("=== fulltext_service ===")
print(repr(data.get("fulltext_service"))[:1000])

print("\n=== page (type, and keys if dict/list) ===")
pg = data.get("page")
print("type:", type(pg))
if isinstance(pg, list) and pg:
    print("first page item keys:", list(pg[0].keys()) if isinstance(pg[0], dict) else pg[0])
    print("first page item sample:", json.dumps(pg[0], indent=1)[:800])
elif isinstance(pg, dict):
    print("keys:", list(pg.keys()))

print("\n=== resource (if present) ===")
res = data.get("resource")
if isinstance(res, dict):
    print("keys:", list(res.keys()))
    for k in ("fulltext","full_text","ocr","text"):
        if k in res: print(f"  resource[{k}] sample:", repr(res[k])[:300])

=== fulltext_service ===
'https://tile.loc.gov/text-services/word-coordinates-service?segment=/service/ndnp/ohi/batch_ohi_goyle_ver01/data/sn87076601/00516993893/1921072701/0279.xml&format=alto_xml&full_text=1'

=== page (type, and keys if dict/list) ===
type: <class 'list'>
first page item keys: ['height', 'info', 'mimetype', 'size', 'url', 'use', 'width']
first page item sample: {
 "height": 6530,
 "info": "https://tile.loc.gov/image-services/iiif/service:ndnp:ohi:batch_ohi_goyle_ver01:data:sn87076601:00516993893:1921072701:0279/info.json",
 "mimetype": "image/jp2",
 "size": 4119103,
 "url": "https://tile.loc.gov/storage-services/service/ndnp/ohi/batch_ohi_goyle_ver01/data/sn87076601/00516993893/1921072701/0279.jp2",
 "use": "",
 "width": 5047
}

=== resource (if present) ===
keys: ['fulltext_file', 'image', 'pdf', 'segment_count', 'url', 'word_coordinates']


In [3]:
import requests
HEADERS = {"User-Agent": "AcademicResearch/1.0 (gender-bias-panel; mailto:shikhadodecha14@gmail.com)"}
url = "http://www.loc.gov/resource/sn87076601/1921-07-27/ed-1/?sp=1&fo=json"
data = requests.get(url, headers=HEADERS, timeout=30).json()

fts = data.get("fulltext_service")
ftf = data.get("resource", {}).get("fulltext_file")
print("fulltext_service URL:", fts)
print("fulltext_file URL   :", ftf)

for label, u in [("fulltext_service", fts), ("fulltext_file", ftf)]:
    if not u: 
        print(f"\n[{label}] missing"); continue
    try:
        r = requests.get(u, headers=HEADERS, timeout=60)
        print(f"\n[{label}] status={r.status_code} bytes={len(r.text)}")
        print("  first 300 chars:", repr(r.text.strip()[:300]))
    except Exception as e:
        print(f"\n[{label}] ERROR: {e}")

fulltext_service URL: https://tile.loc.gov/text-services/word-coordinates-service?segment=/service/ndnp/ohi/batch_ohi_goyle_ver01/data/sn87076601/00516993893/1921072701/0279.xml&format=alto_xml&full_text=1
fulltext_file URL   : https://tile.loc.gov/text-services/word-coordinates-service?segment=/service/ndnp/ohi/batch_ohi_goyle_ver01/data/sn87076601/00516993893/1921072701/0279.xml&format=alto_xml&full_text=1

[fulltext_service] status=200 bytes=23359
  first 300 chars: '{"/service/ndnp/ohi/batch_ohi_goyle_ver01/data/sn87076601/00516993893/1921072701/0279.xml":{"full_text":"t\\nD\\nCimo WEATHER\\u2014Thunderstors this afternees er tomight; somewhat cooler fa the north portion; partly cloudy Thursdays somewhat lower temperature.\\nI\\nI \'1\\nT Bt IT.r\\ncaaweO\\ny6\\u2014H5\\"'

[fulltext_file] status=200 bytes=23359
  first 300 chars: '{"/service/ndnp/ohi/batch_ohi_goyle_ver01/data/sn87076601/00516993893/1921072701/0279.xml":{"full_text":"t\\nD\\nCimo WEATHER\\u2014Thunderstors this af